# Project 43 – AI Risk Review
## Case study: AI-based income prediction and financial decision support

**Research question:** How can an AI-based income prediction system create privacy, bias and misuse risks, and how can these risks be identified, evaluated and mitigated?

**Dataset:** UCI Adult Census Income (1994 Census extraction), supplied as `adult.data` and `adult.test`.

**Important:** This is a classroom case study. The dataset predicts an income category; it is not a real bank-loan or hiring dataset. Any financial decision scenario discussed here is hypothetical. Do not use this model to make real decisions about people.


## 1. Imports and reproducibility
We set a random seed so that the experiment is repeatable.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Load the real UCI dataset
Place `adult.data` and `adult.test` inside `data/raw/` before running this cell.

The official test file starts with a comment line and its labels end with a period, so we handle both details explicitly.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")

columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"
]

train_path = DATA_DIR / "adult.data"
test_path = DATA_DIR / "adult.test"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        "Could not find adult.data and adult.test. "
        "Copy both files into Project43_UCI_Adult_Starter/data/raw/"
    )

train_df = pd.read_csv(
    train_path, header=None, names=columns, skipinitialspace=True,
    na_values="?"
)
test_df = pd.read_csv(
    test_path, header=None, names=columns, skiprows=1,
    skipinitialspace=True, na_values="?"
)

# Remove the trailing full stop used in labels in adult.test.
for frame in (train_df, test_df):
    frame["income"] = frame["income"].astype(str).str.strip().str.rstrip(".")
    frame["income"] = frame["income"].str.strip()

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head())


Training shape: (32561, 15)
Test shape: (16281, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 3. Understand and inspect the data
The target is `income`: `>50K` or `<=50K`. The `fnlwgt` field is a census sampling weight; we will leave it out of this introductory predictive model.

In [3]:
print("Training data information:")
display(train_df.info())
print("\nIncome counts:")
display(train_df["income"].value_counts(dropna=False))
print("\nMissing values in training data (top 10):")
display(train_df.isna().sum().sort_values(ascending=False).head(10))


Training data information:
<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       30725 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education_num   32561 non-null  int64
 5   marital_status  32561 non-null  str  
 6   occupation      30718 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital_gain    32561 non-null  int64
 11  capital_loss    32561 non-null  int64
 12  hours_per_week  32561 non-null  int64
 13  native_country  31978 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


None


Income counts:


income
<=50K    24720
>50K      7841
Name: count, dtype: int64


Missing values in training data (top 10):


occupation        1843
workclass         1836
native_country     583
fnlwgt               0
education            0
education_num        0
age                  0
marital_status       0
relationship         0
sex                  0
dtype: int64

## 4. Prepare features and target
We keep `sex` and `race` in the full model so that we can audit group outcomes. Later, we compare it with a model that excludes those two attributes. Excluding them does not guarantee fairness because other variables can act as proxies.

In [4]:
TARGET = "income"
DROP_COLUMNS = ["income", "fnlwgt"]

X_train = train_df.drop(columns=DROP_COLUMNS)
y_train = train_df[TARGET].map({"<=50K": 0, ">50K": 1})

X_test = test_df.drop(columns=DROP_COLUMNS)
y_test = test_df[TARGET].map({"<=50K": 0, ">50K": 1})

# Remove any rows with unexpected target labels, if present.
train_valid = y_train.notna()
test_valid = y_test.notna()
X_train, y_train = X_train.loc[train_valid], y_train.loc[train_valid].astype(int)
X_test, y_test = X_test.loc[test_valid], y_test.loc[test_valid].astype(int)

numeric_features = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
categorical_features = [c for c in X_train.columns if c not in numeric_features]

def make_preprocessor(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols)
    ])

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Missing values are handled inside the preprocessing pipeline.")


X_train: (32561, 13) | X_test: (16281, 13)
Missing values are handled inside the preprocessing pipeline.


## 5. Baseline model
The DummyClassifier predicts the most frequent training class. It is a simple benchmark, not a useful intelligent predictor.

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

def metric_row(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }

baseline_results = pd.DataFrame([metric_row("Dummy baseline", y_test, baseline_pred)])
display(baseline_results.round(3))


,Model,Accuracy,Precision,Recall,F1-score
0,Dummy baseline,0.764,0.0,0.0,0.0


In [6]:
RANDOM_STATE = 42

## 6. Train the Logistic Regression model
This is the main machine-learning model. The pipeline performs imputation, scaling, one-hot encoding and classification.

In [7]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

full_preprocessor = make_preprocessor(numeric_features, categorical_features)

full_model = Pipeline([
    ("preprocess", full_preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight=None, random_state=RANDOM_STATE))
])

full_model.fit(X_train, y_train)
full_pred = full_model.predict(X_test)
full_results = pd.DataFrame([
    metric_row("Dummy baseline", y_test, baseline_pred),
    metric_row("Logistic Regression", y_test, full_pred)
])
display(full_results.round(3))

print("Classification report:")
print(classification_report(y_test, full_pred, target_names=["<=50K", ">50K"]))


,Model,Accuracy,Precision,Recall,F1-score
0,Dummy baseline,0.764,0.00,0.000,0.000
1,Logistic Regression,0.851,0.73,0.587,0.651


Classification report:
              precision    recall  f1-score   support

       <=50K       0.88      0.93      0.91     12435
        >50K       0.73      0.59      0.65      3846

    accuracy                           0.85     16281
   macro avg       0.80      0.76      0.78     16281
weighted avg       0.84      0.85      0.85     16281



In [8]:
import joblib

joblib.dump(full_model, "../results/income_prediction_model.pkl")

print("Trained model saved successfully!")

Trained model saved successfully!


In [9]:
# Test the trained model using an existing test record

sample_person = X_test.iloc[[0]]

prediction = full_model.predict(sample_person)

if prediction[0] == 1:
    print("Predicted income: >50K")
else:
    print("Predicted income: <=50K")

Predicted income: <=50K


In [10]:
import pandas as pd

def predict_income(age, workclass, education, education_num,
                   marital_status, occupation, relationship,
                   race, sex, capital_gain, capital_loss,
                   hours_per_week, native_country):

    new_person = pd.DataFrame([{
        "age": age,
        "workclass": workclass,
        "education": education,
        "education_num": education_num,
        "marital_status": marital_status,
        "occupation": occupation,
        "relationship": relationship,
        "race": race,
        "sex": sex,
        "capital_gain": capital_gain,
        "capital_loss": capital_loss,
        "hours_per_week": hours_per_week,
        "native_country": native_country
    }])

    prediction = loaded_model.predict(new_person)[0]

    return ">50K" if prediction == 1 else "<=50K"

In [11]:
import joblib

loaded_model = joblib.load(
    "../results/income_prediction_model.pkl"
)

print("Saved model loaded successfully!")

Saved model loaded successfully!


In [12]:
result = predict_income(
    age=35,
    workclass="Private",
    education="Bachelors",
    education_num=13,
    marital_status="Married-civ-spouse",
    occupation="Exec-managerial",
    relationship="Husband",
    race="White",
    sex="Male",
    capital_gain=0,
    capital_loss=0,
    hours_per_week=40,
    native_country="United-States"
)

print("Predicted income:", result)

Predicted income: >50K


In [13]:
loaded_model = joblib.load(
    "../results/income_prediction_model.pkl"
)

new_person = pd.DataFrame([{
    "age": 35,
    "workclass": "Private",
    "education": "Bachelors",
    "education_num": 13,
    "marital_status": "Married-civ-spouse",
    "occupation": "Exec-managerial",
    "relationship": "Husband",
    "race": "White",
    "sex": "Male",
    "capital_gain": 0,
    "capital_loss": 0,
    "hours_per_week": 40,
    "native_country": "United-States"
}])

loaded_prediction = loaded_model.predict(new_person)

print(
    "Prediction from saved model:",
    ">50K" if loaded_prediction[0] == 1 else "<=50K"
)

Prediction from saved model: >50K


## 7. Five-fold cross-validation on training data
The held-out `adult.test` file is not used during cross-validation. Cross-validation happens only on the training file.

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_validate(
    full_model, X_train, y_train, cv=cv,
    scoring={"accuracy": "accuracy", "precision": "precision",
             "recall": "recall", "f1": "f1"},
    n_jobs=-1
)

cv_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Mean CV score": [
        cv_scores["test_accuracy"].mean(),
        cv_scores["test_precision"].mean(),
        cv_scores["test_recall"].mean(),
        cv_scores["test_f1"].mean()
    ],
    "Std. deviation": [
        cv_scores["test_accuracy"].std(),
        cv_scores["test_precision"].std(),
        cv_scores["test_recall"].std(),
        cv_scores["test_f1"].std()
    ]
})
display(cv_summary.round(3))


,Metric,Mean CV score,Std. deviation
0,Accuracy,0.851,0.003
1,Precision,0.734,0.011
2,Recall,0.598,0.011
3,F1-score,0.659,0.008


## 8. Error analysis: inspect at least 10 incorrect predictions
A false positive means the model predicts `>50K` when the actual label is `<=50K`. A false negative means it predicts `<=50K` when the actual label is `>50K`.

In [15]:
error_analysis = X_test.copy()
error_analysis["actual_income"] = y_test.map({0: "<=50K", 1: ">50K"})
error_analysis["predicted_income"] = pd.Series(full_pred, index=X_test.index).map({0: "<=50K", 1: ">50K"})
error_analysis["error_type"] = np.where(
    (y_test.values == full_pred),
    "Correct",
    np.where(full_pred == 1, "False positive", "False negative")
)

wrong_predictions = error_analysis[error_analysis["error_type"] != "Correct"].copy()
print("Total incorrect predictions:", len(wrong_predictions))
display(wrong_predictions.head(10))

# Save a sample of up to 10 errors for the report.
Path("../results").mkdir(exist_ok=True)
wrong_predictions.head(10).to_csv("../results/error_analysis_10.csv", index=False)
print("Saved: results/error_analysis_10.csv")


Total incorrect predictions: 2423


,age,workclass,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,actual_income,predicted_income,error_type
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K,<=50K,False negative
11,36,Federal-gov,Bachelors,13,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,40,United-States,<=50K,>50K,False positive
14,48,Private,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,3103,0,48,United-States,>50K,<=50K,False negative
45,28,Private,Assoc-voc,11,Married-civ-spouse,Prof-specialty,Wife,White,Female,0,0,36,United-States,>50K,<=50K,False negative
47,54,Private,Some-college,10,Married-civ-spouse,Transport-moving,Husband,White,Male,3908,0,50,United-States,<=50K,>50K,False positive
71,46,Private,Assoc-voc,11,Married-civ-spouse,Tech-support,Husband,White,Male,0,0,40,United-States,<=50K,>50K,False positive
87,55,Private,HS-grad,9,Married-civ-spouse,Sales,Husband,White,Male,0,0,56,United-States,>50K,<=50K,False negative
98,59,Private,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,40,United-States,>50K,<=50K,False negative
112,29,Private,Assoc-voc,11,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States,>50K,<=50K,False negative
119,43,Private,Bachelors,13,Divorced,Sales,Unmarried,White,Female,0,0,40,United-States,>50K,<=50K,False negative


Saved: results/error_analysis_10.csv


## 9. Bias audit by sex and race
We compare selection rates and true-positive rates across groups. Here, a model prediction of `>50K` is treated as a positive prediction. These are descriptive group metrics, not proof of discrimination or fairness by themselves.

In [16]:
audit_test = X_test.copy()
audit_test["actual"] = y_test.values
audit_test["predicted"] = full_pred

def group_audit(frame, group_col):
    rows = []
    for group, part in frame.groupby(group_col, dropna=False):
        positive_rate = (part["predicted"] == 1).mean()
        actually_positive = part[part["actual"] == 1]
        tpr = (actually_positive["predicted"] == 1).mean() if len(actually_positive) else np.nan
        rows.append({
            "Group": str(group),
            "Count": len(part),
            "Predicted >50K rate": positive_rate,
            "True positive rate": tpr
        })
    return pd.DataFrame(rows).sort_values("Count", ascending=False)

sex_audit = group_audit(audit_test.assign(sex=X_test["sex"].values), "sex")
race_audit = group_audit(audit_test.assign(race=X_test["race"].values), "race")

print("Audit by sex:")
display(sex_audit.round(3))
print("Audit by race:")
display(race_audit.round(3))


Audit by sex:


,Group,Count,Predicted >50K rate,True positive rate
1,Male,10860,0.249,0.602
0,Female,5421,0.073,0.505


Audit by race:


,Group,Count,Predicted >50K rate,True positive rate
4,White,13946,0.204,0.595
2,Black,1561,0.080,0.486
1,Asian-Pac-Islander,480,0.231,0.617
0,Amer-Indian-Eskimo,159,0.050,0.316
3,Other,135,0.074,0.360


## 10. Compare with a model excluding sex and race
This is a simple mitigation experiment. It does not guarantee fairness because other attributes can correlate with protected characteristics. We compare performance and group metrics rather than assuming the model is fair.

In [17]:
reduced_features = [c for c in X_train.columns if c not in ["sex", "race"]]
reduced_numeric = [c for c in numeric_features if c in reduced_features]
reduced_categorical = [c for c in reduced_features if c not in reduced_numeric]

reduced_model = Pipeline([
    ("preprocess", make_preprocessor(reduced_numeric, reduced_categorical)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
reduced_model.fit(X_train[reduced_features], y_train)
reduced_pred = reduced_model.predict(X_test[reduced_features])

comparison = pd.DataFrame([
    metric_row("Full model (includes sex/race)", y_test, full_pred),
    metric_row("Reduced model (excludes sex/race)", y_test, reduced_pred)
])
display(comparison.round(3))

reduced_audit_test = X_test[["sex", "race"]].copy()
reduced_audit_test["actual"] = y_test.values
reduced_audit_test["predicted"] = reduced_pred
print("Reduced model audit by sex:")
display(group_audit(reduced_audit_test, "sex").round(3))
print("Reduced model audit by race:")
display(group_audit(reduced_audit_test, "race").round(3))


,Model,Accuracy,Precision,Recall,F1-score
0,Full model (includes sex/race),0.851,0.730,0.587,0.651
1,Reduced model (excludes sex/race),0.851,0.732,0.585,0.650


Reduced model audit by sex:


,Group,Count,Predicted >50K rate,True positive rate
1,Male,10860,0.245,0.596
0,Female,5421,0.076,0.525


Reduced model audit by race:


,Group,Count,Predicted >50K rate,True positive rate
4,White,13946,0.201,0.589
2,Black,1561,0.088,0.503
1,Asian-Pac-Islander,480,0.229,0.617
0,Amer-Indian-Eskimo,159,0.088,0.526
3,Other,135,0.111,0.520


## 11. Privacy, misuse and risk register
The dataset is public and historical, but public availability does not remove privacy and ethical responsibilities. Do not attempt to identify individuals. The risks below relate to a hypothetical organization deploying an income prediction system.

In [18]:
risk_register = pd.DataFrame([
    ["Privacy", "Sensitive socioeconomic information may be exposed or accessed without authorization.", 3, 5, "Restrict access, secure storage, minimize data, define retention and deletion."],
    ["Privacy", "Data may be reused for a purpose applicants or individuals did not expect.", 3, 4, "Purpose limitation, transparency and governance."],
    ["Bias", "Prediction rates or error rates may differ across sex groups.", 4, 4, "Measure group outcomes, investigate causes and monitor regularly."],
    ["Bias", "Prediction rates or error rates may differ across race groups.", 4, 5, "Fairness audits, representative evaluation and human review."],
    ["Misuse", "An organization may use income predictions to unfairly exclude people from opportunities.", 3, 5, "Prohibit solely automated high-impact decisions and require review."],
    ["Misuse", "Users may treat an uncertain prediction as a verified fact about an individual.", 4, 4, "Communicate limitations, uncertainty and intended use."]
], columns=["Category", "Risk scenario", "Likelihood (1-5)", "Impact (1-5)", "Mitigation"])

risk_register["Initial risk score"] = risk_register["Likelihood (1-5)"] * risk_register["Impact (1-5)"]
risk_register["Risk level"] = pd.cut(
    risk_register["Initial risk score"],
    bins=[0, 4, 9, 16, 25],
    labels=["Low", "Moderate", "High", "Critical"]
)
display(risk_register)
risk_register.to_csv("../results/risk_register.csv", index=False)


,Category,Risk scenario,Likelihood (1-5),Impact (1-5),Mitigation,Initial risk score,Risk level
0,Privacy,Sensitive socioeconomic information may be exp...,3,5,"Restrict access, secure storage, minimize data...",15,High
1,Privacy,Data may be reused for a purpose applicants or...,3,4,"Purpose limitation, transparency and governance.",12,High
2,Bias,Prediction rates or error rates may differ acr...,4,4,"Measure group outcomes, investigate causes and...",16,High
3,Bias,Prediction rates or error rates may differ acr...,4,5,"Fairness audits, representative evaluation and...",20,Critical
4,Misuse,An organization may use income predictions to ...,3,5,Prohibit solely automated high-impact decision...,15,High
5,Misuse,Users may treat an uncertain prediction as a v...,4,4,"Communicate limitations, uncertainty and inten...",16,High


## 12. Limitations and conclusion
- This dataset is an extraction from the 1994 US Census and may not represent current populations or Sri Lanka.
- It predicts an income category, not loan repayment, creditworthiness or hiring suitability.
- The financial-decision scenario is hypothetical.
- Group differences do not by themselves establish the cause of bias.
- Removing sex and race does not guarantee fairness because proxy variables may remain.
- Risk ratings are qualitative judgments and must be justified in the written report.
- This model must not be used to make real decisions about people.

**Conclusion:** The experiment demonstrates how a real public dataset can be used to build and evaluate an income classifier while examining privacy, bias and misuse risks. Any real deployment would require legal review, stakeholder consultation, security controls, ongoing fairness monitoring and meaningful human oversight.
